# 심화 세션 6차 - 생성 모델 & AI 최적화 사전과제

---

## 1. 생성 모델

### 1-1. 분류 모델 vs 생성 모델

AI 모델은 크게 **분류 모델**과 **생성 모델**로 나눌 수 있음.

| 구분 | 분류 모델 | 생성 모델 |
|------|--------------------------|----------------------|
| 목표 | 입력 데이터가 어떤 클래스인지 **판별** | 데이터의 분포를 학습해 새로운 데이터 **생성** |
| 학습 대상 | $P(Y|X)$ — 입력 X가 주어졌을 때 레이블 Y의 확률 | $P(X)$ 또는 $P(X, Y)$ — 데이터 자체의 분포 |
| 대표 예시 | 이미지 분류, 스팸 탐지, 감성 분석 | 이미지 생성, 텍스트 생성, 음악 생성 |
| 핵심 질문 | "이 이미지는 고양이인가, 개인가?" | "고양이 이미지를 만들어낼 수 있는가?" |

쉽게 말해, 분류 모델은 **"구별하는"** 모델이고 생성 모델은 **"만들어내는"** 모델임.

### 1-2. 생성 모델의 분류

생성 모델은 데이터 분포를 학습하는 방식에 따라 여러 종류로 나뉨.

```
생성 모델
 ├── AE
 ├── VAE
 ├── GAN
 └── 확산 모델
```

#### 📌 AE

**AE**는 입력 데이터를 압축했다가 다시 복원하는 구조의 신경망임.

```
입력 X  →  [인코더]  →  잠재 벡터 z  →  [디코더]  →  복원 X'
```

- **인코더:** 입력 데이터를 저차원의 **잠재 공간** 으로 압축
- **디코더:** 잠재 벡터를 다시 원래 데이터로 복원
- 학습 목표: 복원된 X'가 원본 X와 최대한 같아지도록 학습 (복원 오차 최소화)

**활용:**
- 노이즈 제거
- 차원 축소
- 이상 탐지 -> 복원이 잘 안 되면 이상 데이터로 판단)

> AE는 데이터를 **복원**하는 데 초점이 맞춰져 있어서, 순수한 생성 목적으로는 한계가 있음.  
> 잠재 공간이 연속적이지 않아서 임의의 벡터로 좋은 이미지를 생성하기 어려움.

#### 📌 VAE

**VAE**는 AE의 단점을 보완한 생성 모델임.  
AE가 잠재 벡터를 **하나의 점**으로 압축하는 것과 달리, VAE는 잠재 공간을 **확률 분포**로 표현함.

```
입력 X  →  [인코더]  →  평균(μ), 분산(σ)  →  분포에서 z 샘플링  →  [디코더]  →  생성 X'
```

**핵심 아이디어:**
- 잠재 공간을 **정규 분포(N(0,1))** 에 가깝게 학습
- 덕분에 잠재 공간이 연속적이고 매끄럽게 됨
- 임의의 잠재 벡터 z를 샘플링하면 새로운 데이터 생성 가능

**AE와의 차이:**
| | AE | VAE |
|-|----|-----|
| 잠재 벡터 | 고정된 하나의 점 | 확률 분포 (μ, σ) |
| 새로운 데이터 생성 | 어려움 | 가능 |
| 잠재 공간 | 불연속적 | 연속적 |

**단점:** 생성 이미지가 다소 흐릿한 경향이 있음.

#### 📌 GAN

**GAN**은 2014년 Ian Goodfellow가 제안한 생성 모델로, 두 신경망이 **서로 경쟁**하며 학습하는 구조임

---

**GAN의 구조:**

```
랜덤 노이즈 z
      │
      ▼
┌───────────┐        가짜 이미지
│ Generator │  ──────────────────┐
│           │                    ▼
└───────────┘             ┌─────────────────┐
                          │  Discriminator  │  →  진짜 / 가짜 판별
실제 데이터  ─────────────▶│                │
                          └─────────────────┘
```

- **Generator:** 랜덤 노이즈를 받아 진짜처럼 보이는 가짜 데이터를 생성. 판별자를 속이는 게 목표.
- **Discriminator:** 입력 데이터가 진짜인지 가짜인지 구별. 생성자를 탐지하는 게 목표.

비유하자면, **생성자는 위조지폐범**, **판별자는 경찰** 같은 관계임.  
둘이 경쟁하면서 생성자는 점점 더 정교한 가짜를 만들고, 판별자는 점점 더 잘 구별하게 됨.

---

**GAN의 장단점:**

| 장점 | 단점 |
|------|------|
| 매우 선명하고 고품질의 이미지 생성 가능 | **학습 불안정** — 생성자/판별자 균형 맞추기 어려움 |
| 다양한 도메인에 응용 가능 (이미지, 음성, 영상 등) | **모드 붕괴** — 다양성 없이 비슷한 샘플만 생성하는 문제 |
| 잠재 공간 조작으로 스타일 변환 가능 | 학습 수렴 여부 판단이 어려움 |

> 모드 붕괴란? 생성자가 판별자를 속이는 몇 가지 패턴만 계속 반복 생성하는 현상.  
> 예를 들어, 다양한 얼굴을 생성해야 하는데 비슷한 얼굴만 계속 만들어내는 경우.

#### 📌 확산 모델

**확산 모델**은 최근 가장 주목받고 있는 생성 모델로, Stable Diffusion, DALL-E, Midjourney 등에서 사용됨.  
GAN의 학습 불안정 문제를 해결하고 더 다양하고 고품질의 이미지를 생성할 수 있음

---

**순확산:**

원본 데이터에 **조금씩 노이즈를 추가**하는 과정임.  
T번의 단계를 거치면 완전한 랜덤 노이즈가 됨.

```
원본 이미지  →  노이즈 조금 추가  →  ...  →  완전한 노이즈
    x₀              x₁                           xT
```

이 과정은 **학습할 필요 없이** 수학적으로 정의됨.

---

**역확산:**

순확산의 반대 방향으로, **노이즈에서 원본을 복원**하는 과정임.  
신경망(주로 U-Net)이 각 단계에서 노이즈를 예측하고 제거하는 방법을 학습함.

```
완전한 노이즈  →  노이즈 조금 제거  →  ...  →  깨끗한 이미지
     xT                xT-1                         x₀
```

**학습 목표:** 각 단계에서 "얼마나 노이즈가 섞였는지"를 예측하도록 학습함.  
학습이 끝나면, **완전한 랜덤 노이즈에서 시작해 역확산을 반복**하면 새로운 이미지가 생성됨.

**확산 모델 요약:**

| | 순확산 | 역확산 |
|-|--------|--------|
| 방향 | 이미지 → 노이즈 | 노이즈 → 이미지 |
| 역할 | 학습 데이터 준비 | 실제 생성 단계 |
| 학습 여부 | 고정 (수학적 정의) | 신경망이 학습 |

---

## 2. AI 최적화

모델을 더 잘 학습시키거나, 이미 학습된 모델을 더 잘 활용하기 위한 기법들을 정리

### 2-1. 전이 학습

#### 개념

**전이 학습**은 한 태스크에서 학습한 지식을 다른 태스크에 재사용하는 방법임.

> 비유: 자전거를 탈 줄 알면 오토바이 배우기가 더 쉬운 것처럼,  
> 이미지 분류를 잘 하는 모델은 의료 이미지 분석 태스크에도 빠르게 적응할 수 있음.

처음부터 학습하려면 엄청난 데이터와 시간이 필요하지만,  
전이 학습을 쓰면 **적은 데이터로도 빠르게 좋은 성능**을 낼 수 있음



#### 전이 학습의 분류

| 방식 | 설명 |
|------|------|
| **Feature Extraction** | 사전학습된 모델의 가중치를 **고정**하고, 마지막 분류 레이어만 새로 학습함. 데이터가 매우 적을 때 유리함. |
| **Fine-tuning** | 사전학습 모델의 일부 또는 전체 가중치를 새 태스크 데이터로 **다시 학습**함. 성능이 더 좋지만 데이터가 더 필요함. |
| **Domain Adaptation** | 소스 도메인(학습 데이터)과 타겟 도메인(실제 사용 환경)이 다를 때, 그 차이를 줄이는 방향으로 학습함. |



#### Pre-Training과 Fine-tuning

현대 딥러닝에서 가장 널리 쓰이는 전이 학습 패러다임임.

**Pre-Training:**
- 대규모 데이터(예: 인터넷 전체 텍스트, 수억 장의 이미지)로 모델을 학습시킴
- 특정 태스크가 아닌 **범용적인 특징**을 학습하는 것이 목표
- 엄청난 컴퓨팅 자원과 시간이 필요함 (보통 기업 단위에서 수행)
- 예: BERT 사전학습, GPT 사전학습

**Fine-tuning:**
- 사전학습된 모델을 **특정 태스크에 맞게 추가 학습**시킴
- 상대적으로 적은 데이터와 짧은 시간으로 가능
- 사전학습 모델이 이미 언어/이미지의 일반적인 특징을 알기 때문에 빠르게 수렴함
- 예: BERT를 감성 분석용으로 파인튜닝

```
[Pre-Training]
대규모 데이터 -> 범용 모델 학습 -> 기반 모델: Fine-tuning (감성분석 모델, 번역 모델, 챗봇모델)
```

### 2-2. AI Alignment

**AI Alignment**는 AI 모델이 인간의 의도, 가치관 등에 맞게 동작하도록 조정하는 연구 분야

LLM은 단순히 다음 단어를 예측하도록 학습되기 때문에 사전학습만으로는 사람이 원하는 방식으로 답변하지 않을 수 있음.  
예를 들어 위험한 정보를 그대로 출력하거나 질문의 의도를 무시하고 엉뚱한 답변을 할 수 있음.

이를 해결하기 위해 등장한 대표적인 기법이 **RLHF**



#### 📌 RLHF

**RLHF**는 인간의 피드백을 강화학습에 활용하여 모델이 더 유용하고 안전하게 동작하도록 훈련하는 기법임.  
ChatGPT, Claude 등 현대 LLM 대부분이 이 방식을 사용함.

**학습 과정:**

```
Step 1. SFT
  사람이 직접 작성한 고품질 예시 데이터로 모델을 파인튜닝

Step 2. Reward Model 학습
  모델이 생성한 여러 응답을 사람이 랭킹을 매김
  이 랭킹 데이터로 "좋은 답변"을 점수화하는 보상 모델 학습

Step 3. RL 최적화 (PPO 알고리즘)
  보상 모델의 점수를 높이는 방향으로 LLM을 강화학습으로 최적화
```

**RLHF의 효과:**
- 더 자연스럽고 유용한 답변 생성
- 유해하거나 위험한 내용 생성 감소
- 사용자 의도를 더 잘 파악해서 답변

**한계:**
- 인간 평가자의 편향이 모델에 그대로 반영될 수 있음
- 레이블링 비용이 높음
- 보상 해킹 — 실제로 좋은 답변이 아니라 보상 모델 점수만 높이는 방향으로 학습할 수 있음

### 2-3. 프롬프트 엔지니어링

**프롬프트 엔지니어링**은 LLM에게 더 좋은 출력을 이끌어내기 위해 **입력(프롬프트)을 효과적으로 설계하는 기술**임.  
모델 가중치를 수정하지 않고, 입력 방식만 바꿔서 성능을 높일 수 있음.



**주요 프롬프트 기법:**

**1. Zero-shot Prompting**
- 예시 없이 바로 태스크를 지시하는 방법
```
"다음 문장의 감성을 긍정/부정으로 분류해줘: 오늘 영화 정말 재밌었어!"
```

**2. Few-shot Prompting**
- 몇 가지 예시를 제공해서 모델이 패턴을 파악하게 함
```
"오늘 정말 행복해!" -> 긍정
"내인생은 왜이럴까" -> 부정
"이 영화 최고였어!" -> ??..
```

**3. Chain-of-Thought Prompting**
- 모델이 단계적으로 추론 과정을 작성하도록 유도함
- 복잡한 수학 문제, 논리 추론에서 효과가 큼
```
"차근차근 단계별로 생각해서 답을 구해줘."
```

**4. Role Prompting**
- 모델에게 특정 역할을 부여해 해당 관점에서 답변하게 함
```
"너는 10년 경력의 파이썬 전문가야. 다음 코드를 리뷰해줘."
```

**5. 구조화된 출력 요청**
- 원하는 출력 형식을 명확히 지정함
```
"결과를 JSON 형식으로 출력해줘: {name, age, hobby}"
```

---

**좋은 프롬프트의 공통 원칙:**
- ✅ 태스크를 **구체적**으로 설명
- ✅ 원하는 **출력 형식** 명시
- ✅ **역할** 부여
- ✅ 필요하면 **예시** 포함
- ✅ 복잡한 문제는 **단계별 추론** 요청

## 요약~~~

```
생성 모델
 ├── AE   : 압축 → 복원 구조. 생성보다는 특징 추출/노이즈 제거에 강점
 ├── VAE  : 잠재 공간을 확률 분포로 표현 → 연속적인 생성 가능
 ├── GAN  : 생성자 vs 판별자 경쟁 학습 → 고품질 생성, but 불안정
 └── 확산 : 노이즈 추가(순) → 노이즈 제거(역) 반복 → 안정적 고품질 생성

AI 최적화
 ├── 전이 학습
 │    ├── Pre-Training : 대규모 데이터로 범용 특징 학습
 │    └── Fine-tuning  : 특정 태스크에 맞게 추가 학습
 ├── AI Alignment
 │    └── RLHF : 인간 피드백 → 보상 모델 → 강화학습으로 정렬
 └── 프롬프트 엔지니어링
      └── Zero/Few-shot, CoT, Role 등 다양한 기법으로 성능 향상
```

# 감사합니다. 🦭🦭🦭